# pandas 之五：数据读取与预处理

---

前面几篇讲的是"内存里的数据"。但真实工作时，数据是**躺在文件里**的：
CSV、Excel、TXT、JSON……

这篇就是**从文件拿到干净数据**的完整流程，也是数据分析真正的第一步。

## 数据处理的通用流程

```
① 读取文件  ->  ② 看看长什么样  ->  ③ 处理缺失值
   ->  ④ 处理重复值  ->  ⑤ 转换类型  ->  ⑥ 变形/拆分  ->  ⑦ 开始分析
```

## 学习目标

1. 掌握 `read_csv` / `read_excel` 的**常用参数**
2. 会处理缺失值（`isna` / `dropna` / `fillna`）
3. 会处理重复值（`duplicated` / `drop_duplicates`）
4. 会做类型转换和字段拆分
5. 理解宽表 / 长表的互相转换


In [ ]:
import pandas as pd
import numpy as np
import os
import tempfile

print('pandas 版本：', pd.__version__)


---
# 第一部分：读取文件

## 一、`read_csv` 核心参数速查

| 参数 | 作用 | 常用值 |
|---|---|---|
| `sep` | 分隔符 | `','`（默认）、`'\t'`、`'\s+'` |
| `header` | 哪一行是列名 | `0`（默认）、`None`（无表头） |
| `names` | 自定义列名 | `['A','B']` |
| `index_col` | 哪列当索引 | `0` 或列名 |
| `usecols` | 只读哪几列 | `[0,2]` 或 `['A','C']` |
| `dtype` | 指定列类型 | `{'Age': 'int32'}` |
| `encoding` | 编码 | `'utf-8-sig'`（中文 CSV） |
| `na_values` | 哪些值算缺失 | `['NA','NULL','']` |
| `nrows` | 只读前 N 行 | `1000` |
| `skiprows` | 跳过前 N 行 | `2` |
| `parse_dates` | 解析日期列 | `['date']` |

### ⚠️ 关于编码：中文 CSV 最常见的坑

Windows 上的 Excel 保存的 CSV，**中文经常是 `GBK` 或带 BOM 的 `UTF-8`**。
直接用默认 `utf-8` 读会报 `UnicodeDecodeError`，或者列名前面多出一个怪符号。


In [ ]:
# 先造一份带 BOM 的中文 CSV，用来演示
demo_dir = tempfile.mkdtemp(prefix='pd_demo_')
print('演示文件目录（系统临时目录）：', demo_dir)

csv_path = os.path.join(demo_dir, 'employees.csv')
with open(csv_path, 'w', encoding='utf-8-sig') as f:
    f.write('姓名,部门,工资\n张三,技术部,15000\n李四,市场部,12000\n王五,技术部,18000\n')

print('\n文件已创建。看看原始字节的前 3 个：')
with open(csv_path, 'rb') as f:
    print(f.read(3), '  <- ef bb bf 就是 BOM（字节顺序标记）')


In [ ]:
# 用默认 utf-8 读，列名会带上 BOM 字符
df_bad = pd.read_csv(csv_path)
print('用默认编码读，列名变成：', list(df_bad.columns))
print('  <- 注意第一个列名前面有个看不见的 \\ufeff')

# 正确做法：用 utf-8-sig
df_ok = pd.read_csv(csv_path, encoding='utf-8-sig')
print('\n用 utf-8-sig 读，列名正常：', list(df_ok.columns))


> **`utf-8-sig` 里的 "sig" 就是 BOM 的意思。**
> 这个 `sig` 会主动把 BOM 吃掉。
>
> **记住这个经验**：读 Windows 导出的中文 CSV，
> **`encoding='utf-8-sig'` 应该成为你的默认选择**。
> 读不出来再换 `'gbk'` 试试。


### 演示：`sep` / `header` / `index_col` / `usecols` / `nrows`


In [ ]:
# 造一份"有表头、制表符分隔"的 demo 文件
tsv_path = os.path.join(demo_dir, 'demo.tsv')
with open(tsv_path, 'w', encoding='utf-8') as f:
    f.write('id\tname\tscore\tscore2\n')
    for i in range(1, 6):
        f.write(f'{i}\t学生{i}\t{60 + i}\t{70 + i}\n')

print('=== 1) 默认读（制表符要用 sep=\'\\t\'）===')
print(pd.read_csv(tsv_path, sep='\t'))

print('\n=== 2) usecols 只读两列 ===')
print(pd.read_csv(tsv_path, sep='\t', usecols=['name', 'score']))

print('\n=== 3) nrows 只读前 3 行 ===')
print(pd.read_csv(tsv_path, sep='\t', nrows=3))

print('\n=== 4) index_col 把第 1 列当索引 ===')
print(pd.read_csv(tsv_path, sep='\t', index_col=0))


In [ ]:
# header=None：文件没有表头时，自动用数字 0,1,2... 当列名
noheader_path = os.path.join(demo_dir, 'noheader.csv')
with open(noheader_path, 'w', encoding='utf-8') as f:
    f.write('1,a,100\n2,b,200\n3,c,300\n')

print('header=None（用默认数字列名）：')
print(pd.read_csv(noheader_path, header=None))

print('\nheader=None + names（自己起列名）：')
print(pd.read_csv(noheader_path, header=None, names=['id', '名字', '分数']))


> **原稿提到 `delim_whitespace=True`** —— 这个参数在
> **pandas 3.0 里已经被删除**了，用了会直接报 `TypeError`。
> 替代写法是 **`sep='\s+'`**（正则：一个或多个空白字符）。
> 下面演示正确写法。


In [ ]:
# 多个空格分隔的文件，用正则 \s+
space_path = os.path.join(demo_dir, 'space.txt')
with open(space_path, 'w', encoding='utf-8') as f:
    f.write('A  B   C\n1  2   3\n4  5   6\n')

print(r"sep='\s+' 正确读多空格分隔的文件：")
print(pd.read_csv(space_path, sep=r'\s+'))


---
## 二、读真实数据：员工表

现在用项目里真实的 `employee_data.csv`（500 条员工记录）来演示。

> 这份数据由 `import employee.ipynb` 生成（用 faker 造的假数据）。


In [ ]:
# 这份 CSV 也是带 BOM 的，所以指定 utf-8-sig
employees = pd.read_csv('employee_data.csv', encoding='utf-8-sig')
print('形状：', employees.shape)
print('\n前 5 行：')
employees.head()


**拿到新数据的第一步永远是"看"**：


In [ ]:
print('=== 列名和类型 ===')
employees.info()


In [ ]:
print('=== 数值列的统计概览 ===')
employees.describe()


---
# 第二部分：数据清洗

## 三、缺失值处理

### 2.1 先认识"缺失值"长什么样

pandas 里缺失值有好几个"马甲"，**它们都会被统一识别为缺失**：

| 写法 | 说明 |
|---|---|
| `np.nan` | Numpy 的 NaN（Not a Number） |
| `None` | Python 原生的 None |
| `pd.NA` | pandas 自己的缺失值标记 |
| `pd.NaT` | 时间类型的缺失（Not a Time） |


In [ ]:
s = pd.Series([1, 2, np.nan, None, pd.NA, 5])
print('混了各种缺失值的 Series：')
print(s)
print('\nisna() 能全部识别出来：')
print(s.isna())


### 2.2 造一份带缺失值的数据来练


In [ ]:
df = pd.DataFrame({
    '第一列': [1, 2, np.nan, 4],
    '第二列': [5, np.nan, 7, 8],
    '第三列': [9, 10, 11, 12],
})
print(df)


In [ ]:
print('每个【列】的缺失值个数（axis=0）：')
print(df.isna().sum(axis=0))

print('\n每个【行】的缺失值个数（axis=1）：')
print(df.isna().sum(axis=1))


### 2.3 处理方式一：直接删（`dropna`）

`dropna` 有几个关键参数，不同组合删的东西完全不一样：


In [ ]:
print('原始：')
print(df)

print('\n=== dropna() 默认：只要这行有一个缺失值就删 ===')
print(df.dropna())

print('\n=== how="all"：只有整行全缺失才删（本例没人被删）===')
print(df.dropna(how='all'))

print('\n=== axis=1：删掉"有缺失值的列" ===')
print(df.dropna(axis=1))


In [ ]:
print('=== thresh=2：至少要有 2 个非缺失值，否则删掉这行 ===')
print(df.dropna(thresh=2))

print('\n=== subset=[\'第一列\']：只看这一列，它有缺失就删 ===')
print(df.dropna(subset=['第一列']))


**`dropna` 参数总表**：

| 参数 | 含义 | 默认 |
|---|---|---|
| `axis` | 0=删行，1=删列 | 0 |
| `how` | `'any'`=有缺就删，`'all'`=全缺才删 | `'any'` |
| `thresh` | 保留"至少 n 个非缺失"的行/列 | None |
| `subset` | 只看指定列/行 | None |


> **⚠️ 真实项目建议**：**不要轻易 `dropna()`**！
> 数据很宝贵，一口气删掉可能删掉一半样本。
> **优先考虑填充**，或者至少先算一下 `len(df) - len(df.dropna())` 看看代价。


### 2.4 处理方式二：填充（`fillna`）

这是更常用的做法。填充策略按场景选：

| 策略 | 代码 | 适合什么数据 |
|---|---|---|
| 填 0 | `fillna(0)` | 计数类（没发生就是 0） |
| 填均值 | `fillna(df['c'].mean())` | 近似正态的数值 |
| 填中位数 | `fillna(df['c'].median())` | **有极端值时的首选** |
| 填众数 | `fillna(df['c'].mode()[0])` | 分类/文字 |
| 前向填充 | `ffill()` | 时间序列（延续上一个值） |
| 后向填充 | `bfill()` | 时间序列（用下一个值回填） |


In [ ]:
dff = df.copy()

print('1) 全部填 0：')
print(dff.fillna(0))

print('\n2) 用各列均值填（自动按列对齐）：')
print(dff.fillna(dff.mean()))

print('\n3) 用字典给不同列不同策略：')
print(dff.fillna({'第一列': 0, '第二列': 99}))


In [ ]:
# 4) 众数填充（分类数据常用）
d_mode = pd.DataFrame({'A': [1, 1, np.nan, 2, 1, np.nan], 'B': [np.nan, 2, 3, np.nan, 5, 6]})
print('原始：')
print(d_mode)

mode_value = d_mode['A'].mode()[0]
print('\nA 列的众数 =', mode_value)
print('用众数填充后：')
print(d_mode['A'].fillna(mode_value))


> **为什么众数是 `mode()[0]` 而不是 `mode()`？**
> 因为 `mode()` 返回的是一个 **Series**（可能有多个众数并列），
> 所以要加 `[0]` 取出第一个。**少了这个 `[0]` 是最常见的笔误之一。**


In [ ]:
# 5) 前向/后向填充
d_ff = pd.DataFrame({'A': [1, np.nan, np.nan, 4, np.nan, 6]})
print('原始：')
print(d_ff.T)
print('\nffill 前向填充（把上一个值"拖"下来）：')
print(d_ff.ffill().T)
print('\nbfill 后向填充（用下一个值"顶"上来）：')
print(d_ff.bfill().T)


> **原稿提醒过的一个点**：`fillna(method='ffill')` 这种写法
> **在 pandas 3.0 里已经被移除**，现在必须用 **`.ffill()` / `.bfill()`**。
> 老教程里的写法拿到新版本会报错，这是个很典型的"版本差异"。


In [ ]:
try:
    d_ff.fillna(method='ffill')
except TypeError as e:
    print('fillna(method=...) 已移除 -> 报错：')
    print(str(e)[:160])


## 四、重复值处理

先造一份有重复的数据：


In [ ]:
dup = pd.DataFrame({
    'name': ['Alice', 'Alice', 'Charlie', 'David', 'Eve', 'Frank'],
    'age': [25, 25, 35, 40, 45, 50],
    'city': ['LA', 'LA', 'Chicago', 'Houston', 'Phoenix', 'San Diego'],
})
print(dup)


In [ ]:
# duplicated：标出"这一行是否和前面某行完全一样"
is_dup = dup.duplicated()
print('duplicated()：')
print(is_dup)

print('\n看看哪些行被判为重复：')
print(dup[is_dup], '  <- 第 1 行和第 0 行完全一样')


In [ ]:
print('drop_duplicates() 去重（保留第一条）：')
print(dup.drop_duplicates())

print('\nkeep="last" 保留最后一条：')
print(dup.drop_duplicates(keep='last'))


In [ ]:
# subset：只按指定列判断重复（非常实用）
dup2 = pd.DataFrame({
    '订单号': [1001, 1002, 1003, 1004],
    '客户': ['张三', '张三', '李四', '王五'],
    '金额': [100, 200, 300, 400],
})
print('原始：')
print(dup2)

print('\n按【客户】去重（同一客户只留第一单）：')
print(dup2.drop_duplicates(subset=['客户']))


**`drop_duplicates` 参数表**：

| 参数 | 含义 | 默认 |
|---|---|---|
| `subset` | 只按哪几列判断重复 | 全部列 |
| `keep` | `'first'` / `'last'` / `False`（全删） | `'first'` |
| `ignore_index` | 是否重置索引 | `False` |

> **建议**：去重后加 `.reset_index(drop=True)`，
> 否则索引是"断的"（0,1,3,5...），后面容易出乱子。


In [ ]:
print('去重 + 重置索引：')
print(dup.drop_duplicates().reset_index(drop=True))


## 五、数据类型转换

### 5.1 `astype`：强制转换


In [ ]:
s_num = pd.Series(['1', '2', '3', '4', '5'])
print('原 dtype：', s_num.dtype)
print('转 int 后：', s_num.astype(int).tolist(), ' dtype =', s_num.astype(int).dtype)


### 5.2 转 `category`：省内存 + 加速分组

当一列只有**少数几个固定取值**（性别、省份、状态）时，
`category` 类型会把重复的字符串**只存一份**，用整数指向它。


In [ ]:
s_gender = pd.Series(['男', '女', '男', '女', '男'])

print('object 内存：', s_gender.memory_usage(), '字节')
s_cat = s_gender.astype('category')
print('category 内存：', s_cat.memory_usage(), '字节')
print('类别：', list(s_cat.cat.categories))


数据量大时，这个优化能省几十倍内存。**"文本列转 category" 是标准优化手段。**


### 5.3 `map`：把文字映射成数字

机器学习建模前经常要把"男/女"换成 `1/0`，就用 `map`。


In [ ]:
s_gender = pd.Series(['男', '女', '男', '女', '男'], name='性别')
print('map 成布尔值：')
print(s_gender.map({'男': True, '女': False}))

print('\nmap 成 1/0：')
print(s_gender.map({'男': 1, '女': 0}))


> **原稿错在哪**：原稿写了
> `s_gender['is_male'] = s_gender.map({'男': True, '女': False})`，
> 注释说"会在 DataFrame 中自动新增一列"。
>
> 但 **`s_gender` 是一个 Series，不是 DataFrame** ——
> 对 Series 用 `['is_male'] = ...` 会**往 Series 里加一个名叫 'is_male' 的索引项**，
> 而不是"新增一列"。这在语义上完全是另一回事。
>
> **正确做法**：如果是 Series，就把它 `map` 成新 Series 再赋值给 DataFrame 的新列：
> `df['is_male'] = df['性别'].map({...})`


In [ ]:
# 正确用法演示：在一个 DataFrame 上新增映射列
df_gender = pd.DataFrame({'姓名': ['甲', '乙', '丙'], '性别': ['男', '女', '男']})
df_gender['is_male'] = df_gender['性别'].map({'男': True, '女': False})
print(df_gender)
print('\n新增的列 dtype =', df_gender['is_male'].dtype)


---
# 第三部分：数据变形

## 六、宽表 ↔ 长表

这两个概念很重要：

| 形态 | 长什么样 | 适合什么 |
|---|---|---|
| **宽表**（wide） | 每个科目一列 | **给人看**、做汇报 |
| **长表**（long） | 科目名变成一列的值 | **给机器算**、画图、分组 |

**`melt` 宽变长，`pivot` 长变宽。**


In [ ]:
wide = pd.DataFrame({
    'id': [1, 2],
    'name': ['Alice', 'Bob'],
    'math_score': [85, 90],
    'english_score': [88, 92],
    'science_score': [90, 85],
})
print('原始【宽表】：')
print(wide)


In [ ]:
long = pd.melt(wide, id_vars=['id', 'name'], var_name='subject', value_name='score')
print('melt 变成【长表】：')
print(long)
print('\n形状从', wide.shape, '变成', long.shape)


**`melt` 参数**：

| 参数 | 含义 |
|---|---|
| `id_vars` | **保持不动**的列（相当于"分组依据"） |
| `value_vars` | 要"拉长"的列（不写就是除了 id_vars 之外的全部） |
| `var_name` | 原列名变成的**新列名**（默认叫 `variable`） |
| `value_name` | 原值变成的**新列名**（默认叫 `value`） |


In [ ]:
print("指定 value_vars，只拉长部分列：")
print(pd.melt(wide, id_vars=['id'], value_vars=['math_score'], var_name='科目', value_name='分数'))


### 长表转回宽表：`pivot`


In [ ]:
back = long.pivot(index=['id', 'name'], columns='subject', values='score')
print('pivot 还原成宽表：')
print(back)
print('\n注意索引变成了多级索引（id, name）')


In [ ]:
print('reset_index 把索引变回普通列：')
print(back.reset_index())


> **`pivot` 和 `pivot_table` 的区别**：
> - `pivot`：**不做计算**，行列交叉处必须唯一，否则报错
> - `pivot_table`：**会聚合**（默认求平均），有重复也能处理
>
> 实际做透视表用 `pivot_table` 更多，`pivot` 适合"严格整齐"的数据。


## 七、字段拆分（一列变多列）

场景：`"张三,25"` 想拆成"姓名"和"年龄"两列。


In [ ]:
d_names = pd.DataFrame({'name': ['Alice Smith', 'Bob Johnson']})

d_names[['first_name', 'last_name']] = d_names['name'].str.split(' ', expand=True)
print('str.split 拆成两列：')
print(d_names)


**`expand=True` 是关键**：

| 写法 | 返回 | 用途 |
|---|---|---|
| `.str.split(' ')` | Series（每个元素是列表） | 想保留列表 |
| `.str.split(' ', expand=True)` | **DataFrame** | **想拆成多列** |

少了 `expand=True`，赋值给多个列名会失败。


In [ ]:
splitted = d_names['name'].str.split(' ')
print('不加 expand=True 返回 Series：')
print(splitted)
print('类型：', type(splitted).__name__)

print('\n加 expand=True 返回 DataFrame：')
print(d_names['name'].str.split(' ', expand=True))
print('类型：', type(d_names['name'].str.split(' ', expand=True)).__name__)


### 想"边拆边删原列"，有三种写法


In [ ]:
# 方法 3 最简洁：pop 会"取出并删除"这一列
d_try = pd.DataFrame({'id': [1, 2], 'name': ['Alice Smith', 'Bob Johnson']})
d_try[['first', 'last']] = d_try.pop('name').str.split(' ', expand=True)
print('用 pop 一步到位：')
print(d_try)


**`pop` 的特点：既返回这一列，又把它从原表删掉。** 适合这种"拆完就不要了"的场景。


## 八、数据分箱

把连续数值切成几段，这在"年龄分段""成绩分档"里天天用。


In [ ]:
age_data = pd.DataFrame({'age': [22, 25, 47, 35, 46, 55, 67, 70, 18, 30]})

age_bins = [0, 18, 35, 50, 100]
age_labels = ['少年', '青年', '中年', '老年']

age_data['age_group'] = pd.cut(age_data['age'], bins=age_bins,
                               labels=age_labels, right=False)
print(age_data)

print('\n各年龄段人数：')
print(age_data['age_group'].value_counts().sort_index())


**`right=False` 保证区间是"左闭右开"** `[0,18) [18,35) ...`，
这样 35 岁会归到"中年"，符合直觉。

**口诀：`数值 -> 分箱 -> 统计`，这是数据分析的经典三步。**


## 九、重命名 与 设置索引


In [ ]:
r = pd.DataFrame({'age': [22, 25], 'age_group': ['青年', '青年']})
print('原始列名：', list(r.columns))

print('\nrename 改列名（只改指定的）：')
print(r.rename(columns={'age': '年龄', 'age_group': '年龄组'}))


In [ ]:
si = pd.DataFrame({'id': [1, 2, 3], 'name': ['a', 'b', 'c'], 'score': [80, 90, 85]})
print('原始：')
print(si)

print('\nset_index("id") 把 id 列变成索引：')
print(si.set_index('id'))

print('\nreset_index() 把索引变回普通列：')
print(si.set_index('id').reset_index())


---
## 十、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 读中文 CSV 报 `UnicodeDecodeError` | 读不进 | 先试 `encoding='utf-8-sig'`，再试 `'gbk'` |
| 列名前面多个怪符号 | BOM 没处理 | 用 `utf-8-sig` |
| 用 `fillna(method='ffill')` | pandas 3.0 报 `TypeError` | 改用 `.ffill()` |
| 用 `delim_whitespace=True` | pandas 3.0 报错 | 改用 `sep='\s+'` |
| `mode()` 忘了 `[0]` | 类型不对、后续报错 | 众数是 `mode()[0]` |
| 随手 `dropna()` | 样本被删掉一大半 | 先算代价，优先 `fillna` |
| `str.split` 忘了 `expand=True` | 赋值给多列失败 | 拆多列必须加 |
| 去重后索引是"断的" | 后续按位置取值出错 | 加 `.reset_index(drop=True)` |
| `pivot` 遇到重复组合 | `ValueError` | 改用 `pivot_table` |
| 对 Series 用 `s['新列名']=` 想加列 | 语义完全不对 | 加列要作用于 DataFrame |

---

## 十一、动手练习

**练习 1**：造一个含缺失值的 DataFrame，分别用 ①删行 ②填均值 ③前向填充 三种方式处理，说出各自的适用场景。

**练习 2**：有一列 `['A','A','B','C','B']`，去重并重置索引。

**练习 3**：把 `['1','2','3']` 转成整数类型，并打印转换后的 dtype。

**练习 4**：把这份数据从宽表变成长表，再变回宽表验证能还原：
`{'姓名':['张三','李四'], '数学':[90,88], '英语':[85,92]}`

**练习 5**：把 `'北京市-朝阳区'`、`'上海市-浦东新区'` 这一列按 `-` 拆成"省"和"区"两列。

**练习 6**：把年龄 `[5, 17, 25, 40, 70]` 按 `[0,18,35,60,100]` 分成 `未成年/青年/中年/老年`。

---

**答案思路（先自己做）**

```python
# 1
d = pd.DataFrame({'A': [1, np.nan, 3], 'B': [np.nan, 2, 3]})
print(d.dropna())                 # 删：数据多、缺失少时
print(d.fillna(d.mean()))         # 填均值：数值列
print(d.ffill())                  # 前向填充：时间序列

# 2
print(pd.Series(['A', 'A', 'B', 'C', 'B']).drop_duplicates().reset_index(drop=True))

# 3
s = pd.Series(['1', '2', '3']).astype(int)
print(s.dtype)

# 4
wide = pd.DataFrame({'姓名': ['张三', '李四'], '数学': [90, 88], '英语': [85, 92]})
long = pd.melt(wide, id_vars='姓名', var_name='科目', value_name='成绩')
print(long)
print(long.pivot(index='姓名', columns='科目', values='成绩').reset_index())

# 5
c = pd.DataFrame({'地址': ['北京市-朝阳区', '上海市-浦东新区']})
c[['省', '区']] = c['地址'].str.split('-', expand=True)
print(c)

# 6
a = pd.Series([5, 17, 25, 40, 70])
print(pd.cut(a, bins=[0, 18, 35, 60, 100],
             labels=['未成年', '青年', '中年', '老年'], right=False))
```
